# 04 · เปรียบเทียบโมเดลและ sensitivity (ระยะที่ 5, RQ2)

1. รันเงื่อนไข C ด้วย `MODEL_KEY` ทีละโมเดล (prompt, persona, seed และโรคเหมือนกันทั้งหมด) —
   run ของ `qwen7b` จาก notebook 03 จะถูกใช้ซ้ำอัตโนมัติ (run id เดียวกัน)
2. เปลี่ยนโมเดล: *Runtime → Restart session* → เปลี่ยน `MODEL_KEY` → Run all
3. Sensitivity: เปลี่ยนสมมติฐานทีละตัว (`w_home`, `f_errand`, `f_visit`, ความสามารถ WFH)
4. ส่วนวิเคราะห์ท้าย notebook รันได้โดยไม่ต้องใช้ GPU

> ข้อสรุปจำกัดอยู่กับ **โมเดลและ quantization ที่ทดลอง** ไม่ใช่ "ค่าย" หรือสถาปัตยกรรมทั้งหมด (หัวข้อ 7.2)

In [ ]:
# ---- Setup (same cell in every notebook) ----
import os, sys, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/ThaiGABM")   # where you uploaded the abm folder
    IN_COLAB = True
except ImportError:                                   # running locally (notebooks 00 and 01)
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    IN_COLAB = False

assert (ROOT / "gabm").exists(), f"gabm/ not found in {ROOT}: upload the abm folder to MyDrive/ThaiGABM"
sys.path.insert(0, str(ROOT))
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)

from gabm import load_config
cfg = load_config(ROOT / "config" / "base.yaml")
FIG = ROOT / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "| Colab:", IN_COLAB)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout or "no GPU")

In [ ]:
# Hugging Face token: needed only for gated models (llama8b). Add it in Colab: key icon (Secrets) -> HF_TOKEN
if IN_COLAB:
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF_TOKEN loaded")
    except Exception:
        print("No HF_TOKEN secret: gated models will fail to download")

In [ ]:
MODEL_KEY = "llama8b"             # @param ["typhoon8b", "qwen7b", "llama8b", "openthaigpt7b", "seallm7b", "mistral7b", "qwen1_5b"]
BACKEND = cfg["llm"]["backend"]
SEEDS = cfg["sim"]["seeds"]
DAYS = 180                        # full-epidemic horizon for the model comparison (base.yaml pilot = 30; ~6x the LLM cost)
RUN_LLM = True                    # False: only run the analysis cells
RUN_SENSITIVITY_LLM = False       # LLM sensitivity sweeps (1 seed each), incl. wfh_scale; rule-based sweeps always run
RUN_ORDER_CHECK = False           # action-order robustness: re-run C with reordered action lists (1 seed each)
SENS_SEED = SEEDS[0]

In [ ]:
import copy
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from gabm.sim import run_simulation, contact_model_from_cfg
from gabm.sensitivity import sensitivity_variants, action_order_variants
from gabm.llm import load_backend
from gabm.analysis import load_runs, load_daily, plot_epidemic, plot_action_shares

cfg["sim"]["days"] = DAYS   # here, not in the parameters cell, so a papermill override of DAYS also reaches cfg
personas = pd.read_csv(ROOT / cfg["personas"]["file"])
cm = contact_model_from_cfg(cfg, ROOT)

def variants():
    """One-at-a-time changes: (tag, cfg, contact model, personas). Shared with scripts/run_server.py."""
    return sensitivity_variants(cfg, ROOT, personas, cm)

n_var = sum(1 for _ in variants())
G_main = len(personas) * cfg["sim"]["days"] * len(SEEDS)
print(f"This model: main {G_main:,} generations; LLM sensitivity {n_var} x {len(personas) * cfg['sim']['days']:,} = "
      f"{n_var * len(personas) * cfg['sim']['days']:,} generations (only if RUN_SENSITIVITY_LLM)")

In [ ]:
# Untagged A/B reference at the current DAYS horizon (CPU, seconds): the baselines for the plots below
base_AB = [run_simulation(cfg, c, seed=0, cm=cm, root=ROOT, verbose=False) for c in "AB"]

# Rule-based (condition B) sensitivity: cheap, shows how much the contact assumptions alone move the epidemic
sens_B = [run_simulation(c2, "B", seed=0, cm=cm2, root=ROOT, tag=tag, verbose=False) for tag, c2, cm2, _ in variants()
          if not tag.startswith("sens_wfh")]
print(len(base_AB), "A/B reference runs and", len(sens_B), "rule-based sensitivity runs ready")

In [ ]:
if RUN_LLM:
    backend = load_backend(cfg["llm"], MODEL_KEY, BACKEND)
    for s in SEEDS:
        run_simulation(cfg, "C", seed=s, personas=personas, backend=backend, model_key=MODEL_KEY, cm=cm, root=ROOT)
    if RUN_SENSITIVITY_LLM:
        for tag, c2, cm2, p2 in variants():
            run_simulation(c2, "C", seed=SENS_SEED, personas=p2, backend=backend, model_key=MODEL_KEY, cm=cm2, root=ROOT, tag=tag)

In [ ]:
# Action-order robustness (does visit_family stay ~0% when the action list is reordered?)
if RUN_ORDER_CHECK and RUN_LLM:
    for tag, c2 in action_order_variants(cfg):
        run_simulation(c2, "C", seed=SENS_SEED, personas=personas, backend=backend,
                       model_key=MODEL_KEY, cm=cm, root=ROOT, tag=tag)
    print("order-check runs done")

## วิเคราะห์: เปรียบเทียบโมเดล (รันได้โดยไม่ต้องใช้ GPU)

In [ ]:
runs = load_runs(ROOT)
main = runs[(runs.condition == "C") & (runs.tag == "") & (runs.days == DAYS)]
metrics = ["attack_rate_window", "peak_I", "peak_day", "mean_q_out", "valid_first_try", "fallback_rate", "runtime_s"]
display(main.groupby("model_key")[metrics].agg(["mean", "std"]).round(4))
act_cols = ["unweighted_" + a for a in cfg["actions"]]
shares_tbl = (main.groupby("model_key")[[c for c in act_cols if c in main.columns]].mean()
              .reindex(columns=act_cols).fillna(0.0))
shares_tbl.columns = [c.replace("unweighted_", "") for c in shares_tbl.columns]
display(shares_tbl.round(3))   # an action no model ever picks shows as 0.000, not as a missing column

ref = runs[(runs.condition.isin(["A", "B"])) & (runs.tag == "") & (runs.days == DAYS)]
daily = load_daily(list(main.run_id) + list(ref.run_id), ROOT)
daily["group"] = np.where(daily.condition == "C", daily.model_key, daily.condition)
colors = {"A": "#7f7f7f", "B": "#4C78A8", "typhoon8b": "#72B7B2", "qwen7b": "#F58518", "llama8b": "#54A24B",
          "openthaigpt7b": "#E45756", "seallm7b": "#B279A2", "mistral7b": "#9D755D", "qwen1_5b": "#BAB0AC"}
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
plot_epidemic(daily, group="group", ax=axes[0], y="I", colors=colors); axes[0].set_title("Infectious I by model")
plot_epidemic(daily, group="group", ax=axes[1], y="q_out", colors=colors); axes[1].set_title("q_out by model")
plt.tight_layout(); plt.savefig(FIG / "04_models.png", dpi=150); plt.show()

models = sorted(main.model_key.unique())
fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 3.5), squeeze=False)
for ax, m in zip(axes[0], models):
    plot_action_shares(daily[daily.model_key == m], ax=ax, title=m)
plt.tight_layout(); plt.savefig(FIG / "04_action_shares.png", dpi=150); plt.show()

In [ ]:
# 2x2 design: effect of Thai/SEA tuning within each base family, plus a metrics.csv for the report
from gabm.analysis import export_metrics
FAMILY = {"llama8b": "Llama-3.1-8B", "typhoon8b": "Llama-3.1-8B",
          "qwen7b": "Qwen2.5-7B", "openthaigpt7b": "Qwen2.5-7B", "seallm7b": "Qwen2.5-7B",
          "mistral7b": "Mistral-7B", "qwen1_5b": "Qwen2.5-1.5B"}
TUNING = {"typhoon8b": "thai", "openthaigpt7b": "thai", "seallm7b": "sea",
          "llama8b": "base", "qwen7b": "base", "mistral7b": "base", "qwen1_5b": "base"}
tab = main.copy()
tab["family"] = tab.model_key.map(FAMILY); tab["tuning"] = tab.model_key.map(TUNING)
print("attack_rate by family x tuning (run both columns of a family to fill the 2x2):")
display(tab.pivot_table(index="family", columns="tuning", values="attack_rate_window", aggfunc="mean").round(4))
display(tab.pivot_table(index="family", columns="tuning", values="mean_q_out", aggfunc="mean").round(3))

allruns, p = export_metrics(ROOT)
print(f"wrote {p}  ({len(allruns)} runs, deduped)")

In [ ]:
# Action-order robustness result: a real behaviour barely moves when the action list is reordered
oc = load_runs(ROOT)
oc = oc[(oc.days == DAYS) & (oc.condition == "C") & (oc.seed == SENS_SEED)
        & (oc.tag.str.startswith("order=") | (oc.tag == ""))]
checked = oc[oc.tag.str.startswith("order=")].model_key.unique()
if len(checked):
    act_cols = ["unweighted_" + a for a in cfg["actions"]]
    t = (oc[oc.model_key.isin(checked)].assign(tag=lambda d: d.tag.replace("", "order=base"))
         .set_index(["model_key", "tag"]).sort_index().reindex(columns=act_cols).fillna(0.0))
    t.columns = [c.replace("unweighted_", "") for c in t.columns]
    display(t.round(3))
else:
    print("No order-check runs yet. Set RUN_ORDER_CHECK = True and rerun the run cells.")

## วิเคราะห์: sensitivity

In [ ]:
sens = runs[runs.tag.str.startswith("sens_") & (runs.days == DAYS)].copy()   # keep horizons apart
sens["param"] = sens.tag.str.extract(r"sens_([^=]+)=")[0]
sens["value"] = sens.tag.str.extract(r"=(.+)$")[0].astype(float)
tbl = sens.pivot_table(index=["param", "value"], columns=["condition", "model_key"], values="attack_rate_window", aggfunc="mean")
display(tbl.round(4))